# Ultravox v0.5 (Llama-3.2-1B)

In [1]:
import os
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import numpy as np
import pandas as pd
import transformers
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

/root/autodl-tmp/env/ultravox/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!source /etc/network_turbo

设置成功
注意：仅限于学术用途和加速访问github/huggingface，不承诺稳定性保证


In [3]:
CACHE_DIR    = "/root/autodl-tmp/LLM_Model"
PROJECT_ROOT = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection")
MODEL_ID     = "fixie-ai/ultravox-v0_5-llama-3_2-1b"

os.environ["HF_HOME"] = CACHE_DIR

pipe = transformers.pipeline(
    model=MODEL_ID,
    trust_remote_code=True,
    model_kwargs={"cache_dir": CACHE_DIR},
)

print("Ultravox model loaded.")

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0


Ultravox model loaded.


In [4]:
SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech. You analyze speech "
    "patterns including: word-finding difficulties, semantic paraphasias, empty speech, "
    "reduced syntactic complexity, repetitions, incomplete utterances, and pragmatic "
    "impairments. Based on the audio, classify the speaker."
)

USER_PROMPT = (
    "Listen to this speech sample carefully. Based on the speech characteristics, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)

In [5]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="ultravox_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [6]:
VALID_LABELS = {"Dementia", "Control"}


def classify_audio(wav_path: Path) -> str:
    """Classify a single audio file. Returns raw model response."""
    audio, sr = librosa.load(str(wav_path), sr=16000, mono=True)

    turns = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"<|audio|>\n{USER_PROMPT}"},
    ]

    output = pipe(
        {"audio": audio, "turns": turns, "sampling_rate": sr},
        max_new_tokens=64,
    )
    return output[0]["generated_text"]

In [7]:
def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    predictions, skipped = [], 0

    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]))
            cleaned = raw.strip().strip("'\".,;:!?").capitalize()
            pred = cleaned if cleaned in VALID_LABELS else None
        except Exception as e:
            raw, pred = str(e), None
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1):.4f}")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) :.4f}")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [8]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt", "Pitt", "Pitt_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Pitt"
evaluate_dataset(csv, audio_dir, "Pitt-raw")

[Pitt-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Pitt, exists=True


Pitt-raw:   0%|          | 2/551 [00:01<07:30,  1.22it/s]

  DEBUG [0] session=002-0 raw="string indices must be integers, not 'str'" pred=None
  INVALID [0] session=002-0 true=Control raw="string indices must be integers, not 'str'"
  DEBUG [1] session=002-1 raw="string indices must be integers, not 'str'" pred=None
  INVALID [1] session=002-1 true=Control raw="string indices must be integers, not 'str'"


Pitt-raw:   1%|          | 4/551 [00:02<03:25,  2.66it/s]

  DEBUG [2] session=002-2 raw="string indices must be integers, not 'str'" pred=None
  INVALID [2] session=002-2 true=Control raw="string indices must be integers, not 'str'"
  INVALID [3] session=002-3 true=Control raw="string indices must be integers, not 'str'"


...

Pitt-raw:  33%|███▎      | 181/551 [00:23<00:45,  8.14it/s]

  INVALID [179] session=242-0 true=Control raw="string indices must be integers, not 'str'"
  INVALID [180] session=242-1 true=Control raw="string indices must be integers, not 'str'"


Pitt-raw:  33%|███▎      | 182/551 [00:24<00:48,  7.57it/s]

  INVALID [181] session=242-2 true=Control raw="string indices must be integers, not 'str'"


KeyboardInterrupt: 

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu-raw")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Demucs"
evaluate_dataset(csv, audio_dir, "Pitt-Demucs")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
evaluate_dataset(csv, audio_dir, "Lu-Demucs")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-Denoiser")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
evaluate_dataset(csv, audio_dir, "Lu-Denoiser")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-FRCRN_SE")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Lu-FRCRN_SE")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-MossFormer")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
evaluate_dataset(csv, audio_dir, "Lu-MossFormer")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-Resemble")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
evaluate_dataset(csv, audio_dir, "Lu-Resemble")